# Financial Research Analyst Agent — Interactive Notebook

End-to-end walkthrough of all five project phases:

| Phase | Content |
|---|---|
| 1 | Setup & corpus exploration |
| 2 | Summarisation pipeline |
| 3 | Sentiment & risk agent |
| 4 | Evaluation metrics |
| 5 | Polish — batch run & results table |

> Run cells top-to-bottom. No API server required — the notebook imports `summarize_filing` directly.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on the path
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

---
## Phase 1 — Setup & Corpus Exploration

In [ ]:
from app.loader import load_markdown_as_documents
from app.chunker import chunk_documents

CORPUS_DIR = PROJECT_ROOT / 'corpus' / 'filings'
filing_ids = sorted(p.stem for p in CORPUS_DIR.glob('*.md'))
print(f'Available filings ({len(filing_ids)}):')
for fid in filing_ids:
    print(f'  {fid}')

In [ ]:
# Load a single filing and inspect documents
docs = load_markdown_as_documents(str(CORPUS_DIR / 'HLSR-2024.md'))
print(f'Documents loaded: {len(docs)}')
print()
for doc in docs:
    section = doc['metadata']['section']
    preview = doc['page_content'][:80].replace('\n', ' ')
    print(f'  [{section:20s}] {preview}...')

In [ ]:
# Inspect chunker behaviour
chunks = chunk_documents(docs, max_chars=800)
print(f'Total chunks: {len(chunks)}  (max_chars=800)')
print()
for i, ch in enumerate(chunks):
    sec = ch['metadata'].get('section', '?')
    idx = ch['metadata'].get('chunk_index', 0)
    print(f'  Chunk {i:2d} | section={sec:<20} | chunk_index={idx} | len={len(ch["page_content"])}')

---
## Phase 2 — Summarisation Pipeline

In [ ]:
import json
from app.summarize import summarize_filing

result = summarize_filing('HLSR-2024')
print(json.dumps(result, indent=2))

In [ ]:
# Try LLM mode (requires GROQ_API_KEY in environment)
import os

if os.getenv('GROQ_API_KEY'):
    result_llm = summarize_filing('ACMR-2023')
    print('LLM result:')
    print(json.dumps(result_llm, indent=2))
else:
    print('GROQ_API_KEY not set — using lexicon fallback (set key in .env to enable LLM mode)')
    result_lex = summarize_filing('ACMR-2023')
    print(json.dumps(result_lex, indent=2))

---
## Phase 3 — Sentiment & Risk Agent

In [ ]:
from app.sentiment_risk import (
    score_sentiment,
    extract_financial_metrics,
    label_from_scores,
    extract_risk_snippets,
)

# Score each section individually
docs = load_markdown_as_documents(str(CORPUS_DIR / 'HLSR-2024.md'))
print(f'{"Section":<20} {"POS":>5} {"NEG":>5} {"UNC":>5}')
print('-' * 40)
for doc in docs:
    pos, neg, unc = score_sentiment(doc['page_content'])
    print(f'{doc["metadata"]["section"]:<20} {pos:>5} {neg:>5} {unc:>5}')

In [ ]:
# Extract financial metrics
full_text = '\n\n'.join(
    f"## {d['metadata']['section']}\n{d['page_content']}" for d in docs
)
metrics = extract_financial_metrics(full_text)
print('Financial metrics:', metrics)

In [ ]:
# Top risk snippets via FAISS retriever
from app.retriever import FaissRetriever
from app.chunker import chunk_documents

chunks = chunk_documents(docs, 800)
retriever = FaissRetriever(chunks)
risk_query = 'cybersecurity regulatory breach penalties volatility churn'
top_risk = retriever.query(risk_query, k=3)

print('Top risk chunks (FAISS retrieval):')
for i, ch in enumerate(top_risk, 1):
    section = ch['metadata'].get('section', '?')
    preview = ch['page_content'][:100].replace('\n', ' ')
    print(f'  {i}. [{section}] {preview}')

---
## Phase 4 — Evaluation Metrics

In [ ]:
import re
import json as _json

GOLD_LABELS = _json.loads((PROJECT_ROOT / 'evaluation' / 'gold_labels.json').read_text())

def groundedness(summary, source_text):
    text = ' '.join(summary.get('highlights', []) + summary.get('risks', []))
    tokens = re.findall(r'\w+', text.lower())
    if not tokens:
        return 0.0
    vocab = set(re.findall(r'\w+', source_text.lower()))
    return sum(1 for t in tokens if t in vocab) / len(tokens)

def coherence(summary):
    hi, ri, tone = summary.get('highlights', []), summary.get('risks', []), summary.get('tone', '')
    return len(hi) == 2 and 1 <= len(ri) <= 3 and tone in {'positive', 'neutral', 'cautious'}

# Sample: evaluate 5 filings
sample_ids = ['HLSR-2022', 'HLSR-2024', 'ACMR-2023', 'ZYNT-2023', 'LUMO-2022']
print(f'{"Filing":<12} {"Grd":>6} {"Coh":>5} {"Pred":<10} {"Gold":<10} {"OK"}')
print('-' * 55)
for fid in sample_ids:
    s = summarize_filing(fid)
    src = (CORPUS_DIR / f'{fid}.md').read_text()
    g = groundedness(s, src)
    c = coherence(s)
    pred = s['tone']
    gold = GOLD_LABELS.get(fid, {}).get('tone', '?')
    ok = '✓' if pred == gold else '✗'
    print(f'{fid:<12} {g:>6.2f} {str(c):>5} {pred:<10} {gold:<10} {ok}')

---
## Phase 5 — Full Batch Run & Results Table

In [ ]:
# Run eval_runner directly (saves to evaluation/sample_summary.json)
from evaluation.eval_runner import run_all_evaluations

results = run_all_evaluations()

In [ ]:
# Print aggregate metrics
agg = results['aggregate_metrics']
print('=== Aggregate Evaluation Metrics ===')
print(f"  Average groundedness   : {agg['avg_groundedness']:.2f}")
print(f"  Sentiment accuracy     : {agg['sentiment_accuracy']:.0%}")
print(f"  Coherence pass rate    : {agg['coherence_pass_rate']:.0%}")
print(f"  Total filings          : {agg['total_filings']}")

In [ ]:
# Per-filing summary table
print(f'{"Filing":<12} {"Tone (pred)":<13} {"Tone (gold)":<13} {"Match":<7} {"Grd":<6} {"Coh"}')
print('-' * 65)
for fid in sorted(results['summaries']):
    sa = results['sentiment_agreement'].get(fid, {})
    g  = results['groundedness'].get(fid, 0.0)
    c  = results['coherence'].get(fid, False)
    pred = sa.get('predicted', '?')
    gold = sa.get('gold', '?')
    match = '✓' if sa.get('match') else '✗'
    coh = '✓' if c else '✗'
    print(f'{fid:<12} {pred:<13} {gold:<13} {match:<7} {g:<6.2f} {coh}')

---
## Via REST API (requires running server)

```bash
uvicorn app.server:app --port 9060 --reload
```

In [ ]:
import requests

APP = 'http://localhost:9060'

try:
    health = requests.get(f'{APP}/health', timeout=3).json()
    print('Server health:', health)
    resp = requests.post(f'{APP}/summarize', json={'filing_id': 'HLSR-2024'}, timeout=10)
    print(json.dumps(resp.json(), indent=2))
except requests.exceptions.ConnectionError:
    print('Server not running — start with: uvicorn app.server:app --port 9060 --reload')